## Experiment Tracking with MLflow

When you train multiple models, it becomes hard to remember which configuration produced which result. **MLflow** solves this by logging parameters, metrics, and artifacts for every run, making experiments **reproducible and comparable**.

### Objective 
You will train four classifiers on the iris dataset and use MLflow to log, organize and compare your experiments

By the end of this lab you will be able to:
1. Create and manage MLflow experiments and runs 
2. Log params, metrics,artifacts 
3. Use nested runs to compare and organize hyperparam sweeps
4. Use MLflow UI to compare and justify a choice


### Key concepts

| Concept | What it is |
|---------|-----------|
| **Experiment** | A named group of related runs |
| **Run** | One execution — one model training |
| **Parameter** | An input setting (e.g. `C=1.0`) |
| **Metric** | A numerical result (e.g. `val_accuracy=0.87`) |
| **Artifact** | A file output (e.g. a confusion matrix plot) |
| **Nested run** | A child run inside a parent k=1, k=5, k=20 grouped under "knn-sweep" |


### Viewing results

After completing this section, open a terminal and run:

```bash
mlflow ui
```

Then open `http://localhost:5000` to compare all your runs visually.


# Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import mlflow
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')


### Load and split the Iris dataset

In [2]:
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names   = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print(f"Train : {X_train.shape[0]} samples")
print(f"Val   : {X_val.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")
print(f"Classes: {list(class_names)}")


Train : 90 samples
Val   : 30 samples
Test  : 30 samples
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


### Helper functions
These functions are to help you get model metrics and save confusion matrix
We'll use them throughout the notebook to log different models and compare between them



In [3]:
def get_metrics(model, X, y):
    """Returns accuracy, macro precision, and macro recall as a dict."""
    y_pred = model.predict(X)
    return {
        "accuracy":  accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y, y_pred, average="macro", zero_division=0),
    }


def save_confusion_matrix(model, X, y, title, path):
    """Saves a confusion matrix plot to disk and returns the file path."""
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return path


---
## Part 1 — Create your Experiment

**TODO 1.1** — Set the MLflow tracking URI and create an experiment called `"classifier-comparison"`.  
If it already exists, reuse it instead of creating a duplicate.


### Setting the MLflow Tracking URI

This step configures **where MLflow stores experiment data** such as parameters, metrics, and artifacts.



In [17]:
import mlflow
from pathlib import Path
mlflow.set_tracking_uri(uri=(Path.cwd() / 'mlruns').as_uri())

### Dealing with Experiments

In [18]:
from mlflow.tracking import MlflowClient

# Initialize the MLflow client
client = MlflowClient()

# Cxistsheck if the experiment already e
exp = client.get_experiment_by_name("classifier-comparison")

if exp is None:
    # Create a new experiment with specified name and tags
    experiment_id = client.create_experiment(
        name="classifier-comparison",
        tags={"topic":"classification-lab", "version":"v1"}
    )
else:
    experiment_id = exp.experiment_id

print("Experiment ID:", experiment_id)

Experiment ID: 151001915503869778


# Dealing with Runs

---
## Part 2 — Logistic Regression Run

 — Train a Logistic Regression model, then log it as an MLflow run.

Inside the run, log:
- Parameters: `C`, `penalty`, `solver`
- Metrics: `val_accuracy`, `val_precision`, `val_recall`
- Artifact: confusion matrix on the validation set


In [6]:
# ── Step 1: Train the model normally
lr = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=200, random_state=42)
lr.fit(X_train, y_train)

# ── Step 2: Open a run and log everything inside it 
with mlflow.start_run(experiment_id=experiment_id, run_name="logistic-regression") as run:

    # Log parameters — the settings you chose before training
    mlflow.log_param("C",       1.0)
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("solver",  "lbfgs")

    # Compute metrics on the validation set and log them
    metrics = get_metrics(lr, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])

    # Save the confusion matrix to a file, then log the file as an artifact
    path = save_confusion_matrix(lr, X_val, y_val,
                                 title="Logistic Regression — Validation",
                                 path="cm_logistic.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices")

    # Store the run_id so we can reference this run later
    lr_run_id = run.info.run_id
    print("Run logged successfully")
    print("Run ID   :", lr_run_id)
    print("Metrics  :", metrics)


Run logged successfully
Run ID   : 03b341ea2ec2454fb87d367719eff6fa
Metrics  : {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333332, 'recall': 0.9333333333333332}


**Q2.1** — You logged three metrics: accuracy, precision, and recall.  
For a balanced dataset like Iris, which single metric is most informative and why?  
Would your answer change if one species were 10× more common than the others?

*Your answer here:*
1. accuracy, data is balanced

2. recall



---
## Part 3 — kNN Sweep 
You will train kNN with three values of k and log them as **nested runs** under a single parent.  
Nested runs keep the UI clean — the sweep appears as one collapsible group rather than three unrelated rows.

**TODO 3.1** — Complete the nested run block below.  
For each k, log: `n_neighbors`, `val_accuracy`, `val_precision`, `val_recall`, and the confusion matrix.


In [9]:
k_values = [1, 5, 15]
knn_models = {}

# The parent run groups all k values together in the UI
with mlflow.start_run(experiment_id=experiment_id, run_name="knn-sweep") as parent_run:

    for k in k_values:

        # Train the model
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)
        knn_models[k] = knn

        # Each k gets its own child run nested under the parent
        with mlflow.start_run(
            experiment_id=experiment_id,
            run_name=f"knn-k{k}",
            parent_run_id=parent_run.info.run_id,
            nested=True
        ) as child_run:

            # TODO: log n_neighbors parameter
            mlflow.log_param("N", k)
            

            # TODO: compute metrics with get_metrics() and log all three
            # YOUR CODE HERE
            metrics = get_metrics(knn, X_val, y_val)
            mlflow.log_metric("val_accuracy",  metrics["accuracy"])
            mlflow.log_metric("val_precision", metrics["precision"])
            mlflow.log_metric("val_recall",    metrics["recall"])            
            
            
            # TODO: save and log the confusion matrix artifact
            # Hint: path = save_confusion_matrix(..., path=f"cm_knn_k{k}.png")
            #       mlflow.log_artifact(path, artifact_path="confusion_matrices")
            # YOUR CODE HERE
            path = save_confusion_matrix(knn, X_val, y_val,
                                        title="KNN — Validation",
                                        path=f"cm_knn_k{k}.png")
            mlflow.log_artifact(path, artifact_path="confusion_matrices")


            
            print(f"  k={k} logged | run_id: {child_run.info.run_id}")
            print("Metrics  :", metrics)


  k=1 logged | run_id: ec97dc816cfd4d17bf393f57dbb28dc9
Metrics  : {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333332, 'recall': 0.9333333333333332}
  k=5 logged | run_id: d8d4a53e17914489bec7c7c9d7737dc7
Metrics  : {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333332, 'recall': 0.9333333333333332}
  k=15 logged | run_id: 0d94bd75932e42c084bd2a8699d749af
Metrics  : {'accuracy': 0.9666666666666667, 'precision': 0.9696969696969697, 'recall': 0.9666666666666667}


---
## Part 4 — SVM Sweep

**TODO 4.1** — Train a linear and an RBF SVM, log them as nested runs under `"svm-sweep"`.  
For each, log: `kernel`, `C`, all three validation metrics, and the confusion matrix.


In [10]:
svm_configs = {
    "svm-linear": SVC(kernel="linear", C=1.0, probability=True, random_state=42),
    "svm-rbf":    SVC(kernel="rbf",    C=1.0, gamma="scale", probability=True, random_state=42),
}

with mlflow.start_run(experiment_id=experiment_id, run_name="svm-sweep") as parent_run:

    for run_name, svm in svm_configs.items():

        # TODO: fit the model
        # YOUR CODE HERE
        svm.fit(X_train, y_train)

        
        with mlflow.start_run(
            experiment_id=experiment_id,
            run_name=run_name,
            parent_run_id=parent_run.info.run_id,
            nested=True
        ) as child_run:

            # TODO: log kernel and C
            # YOUR CODE HERE
            mlflow.log_param("C", svm.C)
            
            # TODO: log confusion matrix artifact
            # YOUR CODE HERE
            path = save_confusion_matrix(svm, X_val, y_val,
                                        title="SVM — Validation",
                                        path=f"{run_name}.png")
            mlflow.log_artifact(path, artifact_path="confusion_matrices")            
            
            print(f"  {run_name} logged | run_id: {child_run.info.run_id}")


  svm-linear logged | run_id: 22259fed5b0145d3a02af82a23984936
  svm-rbf logged | run_id: 67aab4b888c9427e8a5e8fe099af8a3a


---
## Part 5 — Random Forest with Grid Search

**TODO 5.1** — Run a small grid search, then log the best configuration as a single run.  
Log: best params, best CV score, validation metrics, all feature importances, and the confusion matrix.


In [11]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [3, 5, None],
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))


Best params : {'max_depth': 3, 'n_estimators': 50}
Best CV acc : 0.9778


In [13]:
with mlflow.start_run(experiment_id=experiment_id, run_name="random-forest-best") as run:

    # TODO: log best params as a dict in one call
    # Hint: mlflow.log_params(grid_search.best_params_)
    # YOUR CODE HERE
    mlflow.log_params(grid_search.best_params_)
    # TODO: log the best CV score as "cv_accuracy"
    # YOUR CODE HERE
    mlflow.log_metric("cv_accuracy",  grid_search.best_score_)
    
    

    # TODO: compute and log validation metrics
    # YOUR CODE HERE
    metrics = get_metrics(grid_search.best_estimator_, X_val, y_val)
    mlflow.log_metric("val_accuracy",  metrics["accuracy"])
    mlflow.log_metric("val_precision", metrics["precision"])
    mlflow.log_metric("val_recall",    metrics["recall"])      
    

    # log each feature importance as a separate metric

    for name, importance in zip(feature_names, best_rf.feature_importances_):
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        mlflow.log_metric(f"importance_{clean_name}", importance)

    # TODO: log confusion matrix artifact
    # YOUR CODE HERE
    path = save_confusion_matrix(grid_search.best_estimator_, X_val, y_val,
                                title="RFWGS — Validation",
                                path=f"RFWGS.png")
    mlflow.log_artifact(path, artifact_path="confusion_matrices") 
    print("Metrics  :", metrics)
          

    
    # Another way to do this is to plot and log feature importance as artifact
    # Here we log the figure itself as an artifact instead of logging 
    # each importance as a separate metric
    fig, ax = plt.subplots(figsize=(6, 4))
    clean_names = [n.replace("(cm)", "").strip() for n in feature_names]
    ax.barh(clean_names, best_rf.feature_importances_, color="steelblue", edgecolor="k")
    ax.set_xlabel("Importance")
    ax.set_title("Random Forest — Feature Importances")
    plt.tight_layout()

    path = "feature_importances.png"
    fig.savefig(path)
    mlflow.log_figure(fig, artifact_file="feature_importances.png")
    plt.close(fig)


    
    rf_run_id = run.info.run_id
    print("Run ID:", rf_run_id)


Metrics  : {'accuracy': 0.9333333333333333, 'precision': 0.9333333333333332, 'recall': 0.9333333333333332}
Run ID: dcdb74222dd6469cb84acb45188fd747


Look at the feature importances you logged in the UI.  
Which feature contributes most to the classification? 

*Your answer here:*
- petal width


---
## Evaluation

**TODO 7.1** — Evaluate your best model from each family on the **test set** and log everything  
into a single summary run called `"final-test-evaluation"`.  
Use a prefix per model to avoid metric name collisions (e.g. `"lr_test_accuracy"`).


In [14]:
# Replace None with your best SVM model from Part 4
best_svm = svm_configs["svm-rbf"]

best_models = {
    "lr":  lr,
    "knn": knn_models[5],
    "svm": best_svm,
    "rf":  best_rf,
}

with mlflow.start_run(experiment_id=experiment_id, run_name="final-test-evaluation") as run:

    for name, model in best_models.items():

        # TODO: compute test metrics and log with prefix
        # e.g. mlflow.log_metric(f"{name}_test_accuracy", ...)
        # YOUR CODE HERE
        metrics = get_metrics(model, X_val, y_val)
        mlflow.log_metric(f"{name}_test_accuracy", metrics['accuracy'])
        
        # log confusion matrix artifact for each model
        fig, ax = plt.subplots(figsize=(5, 4))
        cm = confusion_matrix(y_test, model.predict(X_test))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues")
        ax.set_title(f"{name} — Test Set")
        plt.tight_layout()
        mlflow.log_figure(fig, artifact_file=f"confusion_matrices/cm_{name}_test.png")
        plt.close(fig)

    print("Run ID:", run.info.run_id)

Run ID: 56efca785d094f5386c609840b06f78a


---
## Part 6 — Autologging

So far you logged everything manually — params, metrics, artifacts, one line at a time.  
MLflow provides `mlflow.sklearn.autolog()` which does all of this automatically the moment you call `.fit()`.
Autolog is generally faster and more complete but manual logging gives you better control

**Read and run the cell below** — Then answer the questions.


In [15]:
# Enable autologging — must be called before .fit()
mlflow.sklearn.autolog()

with mlflow.start_run(experiment_id=experiment_id, run_name="autolog-logistic") as run:
    # Just train normally — MLflow intercepts .fit() and logs everything
    lr_auto = LogisticRegression(C=0.5, penalty="l2", solver="lbfgs", max_iter=200, random_state=42)
    lr_auto.fit(X_train, y_train)
    print("Run ID:", run.info.run_id)

# Turn autologging off so it doesn't interfere with the rest of the lab
mlflow.sklearn.autolog(disable=True)


Run ID: c28be3da7f4e49b4bb6912ac3e6d28d9


Open the MLflow UI and compare the `autolog-logistic` run to your `logistic-regression` run from Part 2.

**Q6.1** — List three things autolog logged that you also logged manually in Part 2.  
Then list one thing autolog logged that you did *not* log manually.


## Part 7 — Final Model Comparison

All four models are trained. Now we evaluate each on the **test set** for the first time
and log the results as nested runs under a single parent called `model-comparison`.

In the MLflow UI:
1. Expand the `model-comparison` group
2. Check all 4 child runs
3. Click **Compare** — you get a side-by-side table and bar chart of all three metrics

In [16]:
# Here we log each model's test performance as nested runs under a single
# parent, so we can use the MLflow UI Compare button to see them side by side.

with mlflow.start_run(experiment_id=experiment_id, run_name="model-comparison") as parent_run:

    for name, model in best_models.items():

        with mlflow.start_run(
            experiment_id=experiment_id,
            run_name=name,
            parent_run_id=parent_run.info.run_id,
            nested=True
        ) as child_run:

            # Evaluate on the test set — this is the first time we touch test data
            metrics = get_metrics(model, X_test, y_test)
            mlflow.log_metric("test_accuracy",  metrics["accuracy"])
            mlflow.log_metric("test_precision", metrics["precision"])
            mlflow.log_metric("test_recall",    metrics["recall"])

            print(f"  {name} | accuracy: {metrics['accuracy']:.3f}")

# To compare: open MLflow UI → expand model-comparison → 
# select all 4 child runs → click Compare

  lr | accuracy: 0.967
  knn | accuracy: 0.967
  svm | accuracy: 0.967
  rf | accuracy: 0.933


In [ ]:
!mlflow ui

**Q7.1 — Model selection**  
Based on the test results you just logged, which model would you choose for production?  
Justify your answer 
*Your answer here:*

---

**Q7.2 — Val vs. Test**  
You evaluated on the validation set throughout Parts 2–5 and only touched the test set here in Part 7.  
What could go wrong if you had used test performance to pick your best k in Part 3?

*Your answer here:*
this should not be done, as this the test set should not be seen


---
## References

- [MLflow documentation](https://mlflow.org/docs/latest/index.html)
- [mlflow.sklearn.autolog](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html#mlflow.sklearn.autolog)
- [scikit-learn: Iris dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
